In [ ]:
from collections import Counter
from pathlib import Path

import torch
from torch import nn

from helpers import get_model_and_transform

class PatchClassifier(nn.Module):
    def __init__(self, backbone: nn.Module, feature_dim: int, dropout: float):
        super().__init__()
        self.backbone = backbone
        self.classifier = nn.Sequential(
            nn.LayerNorm(feature_dim),
            nn.Dropout(dropout),
            nn.Linear(feature_dim, 2),
        )

    def forward(self, x):
        features = self.backbone(x)
        return self.classifier(features)

In [10]:
checkpoint_path = Path('runs0708/virchow2_finetune.pt')
assert checkpoint_path.exists(), f'Missing checkpoint: {checkpoint_path}'

checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
backbone, _ = get_model_and_transform(checkpoint['model_name'])
feature_dim = checkpoint['model_state_dict']['classifier.2.weight'].shape[1]
dropout = checkpoint['args']['dropout']

finetuned_model = PatchClassifier(backbone=backbone, feature_dim=feature_dim, dropout=dropout)
finetuned_model.load_state_dict(checkpoint['model_state_dict'])
finetuned_model = finetuned_model.to(device)
finetuned_model.eval()

print(f"Loaded {checkpoint['model_name']} on {device}")
print(f"Best validation accuracy: {checkpoint['best_val_accuracy']:.4f}")

Loaded virchow2 on cuda
Best validation accuracy: 0.9592


In [11]:
# eval the finetuned model on /mnt/Z/cuhk_data/HPACG/batch1/data

from PIL import Image, ImageFile, PngImagePlugin

Image.MAX_IMAGE_PIXELS = None
PngImagePlugin.MAX_TEXT_CHUNK = 100 * 1024 * 1024
PngImagePlugin.MAX_TEXT_MEMORY = 100 * 1024 * 1024
ImageFile.LOAD_TRUNCATED_IMAGES = True

data_dir = Path('/mnt/Z/cuhk_data/HPACG/batch1/data')

_, transform = get_model_and_transform('virchow2')

preds = []

for positive_png_fp in data_dir.glob('**/positive/*.png'):
    img = Image.open(positive_png_fp).convert('RGB')
    img = transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = finetuned_model(img)
        pred = torch.argmax(logits, dim=1).item()

    preds.append(pred)

print(f'Processed {len(preds)} positive tiles')
print(Counter(preds))

Processed 4309 positive tiles
Counter({1: 4253, 0: 56})


In [12]:
# process negative tiles

neg_preds = []

for negative_png_fp in data_dir.glob('**/negative/*.png'):
    img = Image.open(negative_png_fp).convert('RGB')
    img = transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = finetuned_model(img)
        pred = torch.argmax(logits, dim=1).item()

    neg_preds.append(pred)  

print(f'Processed {len(neg_preds)} negative tiles')
print(Counter(neg_preds))

Processed 2972 negative tiles
Counter({0: 2731, 1: 241})


In [13]:
from draw_attn import draw_attention_mask
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

attn_weights_path = Path('runs0708/virchow2_backbone_best.pt')
assert attn_weights_path.exists(), f'Missing backbone checkpoint: {attn_weights_path}'

attn_output_dir = Path('runs0708/positive_attn_maps')
attn_output_dir.mkdir(parents=True, exist_ok=True)

attn_model, attn_transform = get_model_and_transform('virchow2', weights_path=str(attn_weights_path))
attn_model = attn_model.to('cpu')
attn_model.eval()

positive_png_paths = sorted(data_dir.glob('**/positive/*.png'))
max_positive_images = None  # Set to a small integer for a smoke test.
paths_to_process = positive_png_paths if max_positive_images is None else positive_png_paths[:max_positive_images]

print(f'Attention model ready on cpu from {attn_weights_path}')
print(f'Positive PNGs queued for attention extraction: {len(paths_to_process)}')
print(f'Output directory: {attn_output_dir}')

Attention model ready on cpu from runs0708/virchow2_backbone_best.pt
Positive PNGs queued for attention extraction: 4309
Output directory: runs0708/positive_attn_maps


In [14]:
saved_attention_maps = []

for positive_png_fp in tqdm(paths_to_process, desc='positive attention maps', unit='img'):
    rel_parent = positive_png_fp.parent.relative_to(data_dir)
    out_dir = attn_output_dir / rel_parent
    out_dir.mkdir(parents=True, exist_ok=True)

    out_fp = out_dir / f'{positive_png_fp.stem}_attn.png'
    draw_attention_mask(
        attn_model,
        attn_transform,
        positive_png_fp,
        'virchow2',
        title=positive_png_fp.name,
        save_fp=out_fp,
    )
    plt.close('all')

    saved_attention_maps.append(out_fp)

print(f'Saved {len(saved_attention_maps)} attention maps')
print(saved_attention_maps[:3])

positive attention maps:   0%|          | 0/4309 [00:00<?, ?img/s]

: 